# 01 Simple Baselines


In [27]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parents[0]  # .../spring_semester
workspace_root = project_root.parents[0]        # .../coursework
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.config import ExperimentConfig
from src.train.runner import run_experiment

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
M = 20
N = 100
J = 2
k = 0.4
noise_std = 0.1
seed = 42
model_list = ['counterfactual_residual_baseline', 'logistic_regression']

model_params = {
    'counterfactual_residual_baseline': {'threshold_z': 0.75},
    'logistic_regression': {'lr': 0.05, 'n_iter': 1200, 'l2': 1e-4, 'threshold': 0.5},
}


In [29]:
config = ExperimentConfig(
    M=M,
    N=N,
    J=J,
    k=k,
    model_list=model_list,
    seed=seed,
    noise_std=noise_std,
    base_dir=str(workspace_root),
    data_csv_path='spring_semester/data/data_prepared.csv',
    output_data_dir='spring_semester/data',
    runs_dir='spring_semester/runs',
    model_params=model_params,
)

result = run_experiment(config)
print('Run ID:', result['run_id'])
print('Run dir:', result['run_dir'])
print('Metrics file:', result['metrics_path'])
print('Predictions file:', result['predictions_path'])


Run ID: run_20260503_211007
Run dir: C:\Users\akasa\pet_projects\coursework\spring_semester\runs\run_20260503_211007
Metrics file: C:\Users\akasa\pet_projects\coursework\spring_semester\runs\run_20260503_211007\metrics.csv
Predictions file: C:\Users\akasa\pet_projects\coursework\spring_semester\runs\run_20260503_211007\predictions.csv


In [30]:
metrics_df = pd.read_csv(result['metrics_path'])
pred_df = pd.read_csv(result['predictions_path'])

display(metrics_df.sort_values('f1', ascending=False))
display(pred_df.tail(10))


,accuracy,precision,recall,f1,tn,fp,fn,tp,fp_count,fn_count,model,split,runtime_train_sec,runtime_predict_sec
1,0.986667,1.000000,0.866667,0.928571,270,0,4,26,0,4,logistic_regression,test,0.097524,0.001529
0,0.923333,0.589744,0.766667,0.666667,254,16,7,23,16,7,counterfactual_residual_baseline,test,0.000002,0.341858


,sample_id,group_id,meter_id,label,split,model,score,prediction
590,group_00077_31,group_00077,31,0,test,logistic_regression,0.027396,0
591,group_00077_28,group_00077,28,0,test,logistic_regression,0.023270,0
592,group_00077_8,group_00077,8,0,test,logistic_regression,0.006339,0
593,group_00077_21,group_00077,21,0,test,logistic_regression,0.010193,0
594,group_00077_9,group_00077,9,0,test,logistic_regression,0.089440,0
595,group_00077_6,group_00077,6,0,test,logistic_regression,0.030016,0
596,group_00077_0,group_00077,0,0,test,logistic_regression,0.016024,0
597,group_00077_2,group_00077,2,0,test,logistic_regression,0.009091,0
598,group_00077_4,group_00077,4,0,test,logistic_regression,0.010588,0
599,group_00077_5,group_00077,5,0,test,logistic_regression,0.006082,0


In [ ]:
err = pred_df.copy()
err['is_fp'] = ((err['label'] == 0) & (err['prediction'] == 1)).astype(int)
err['is_fn'] = ((err['label'] == 1) & (err['prediction'] == 0)).astype(int)
summary = err.groupby('model', as_index=False)[['is_fp', 'is_fn']].sum()
display(summary)


,model,is_fp,is_fn
0,counterfactual_residual_baseline,16,7
1,logistic_regression,0,4
